# Phase 2: PII Defense Generation and Alignment
This notebook scales up the attack phase to generate a comprehensive dataset of PII extraction vulnerabilities, and then trains a DPO-aligned model to defend against them.

In [ ]:
!git clone https://github.com/no1ceboy/VDT-PII-Defense.git
%cd VDT-PII-Defense
!pip install -r requirements.txt
!pip install accelerate bitsandbytes peft trl datasets torch torchvision torchaudio transformers wandb

## 1. Setup API Keys & WandB
Load API keys and weights & biases for training metric tracking.

In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ['OPENROUTER_API_KEY'] = user_secrets.get_secret("OPENROUTER_API_KEY")
os.environ['GOOGLE_API_KEY'] = user_secrets.get_secret("GOOGLE_API_KEY")

# Log into WandB for metrics tracking
try:
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    print("Successfully logged into WandB!")
except Exception:
    print("WANDB_API_KEY not found in secrets. Metrics will not be logged online.")

!python -m src.run_attack --samples 50 --positions beginning,middle,end

## 2. Prepare DPO Dataset
Extract successful PII attacks and format them into chosen/rejected preference pairs.

In [ ]:
!python -m src.prepare_dpo_data

## 3. Train the DPO RL Agent
Train a local Qwen 1.5B model using 4-bit QLoRA to align it against PII extractions.

In [ ]:
!python src/train_defense.py